In [1]:
# VectorDB에 이미지 자료를 I/O
# 이미지 벡터화 처리는 Resnet 모델 사용
# 이미지 -> 벡터DB -> 저장 -> 검색
!pip install chromadb sentence-transformers torchvision torch pillow

In [4]:
import os
import torch
from torchvision import transforms
from PIL import Image
from chromadb import PersistentClient
from torchvision.models import resnet18

import torchvision.models as models
all_models = dir(models)
resnet_models = [name for name in all_models if 'resnet' in name.lower()]
print(resnet_models)

['ResNet', 'ResNet101_Weights', 'ResNet152_Weights', 'ResNet18_Weights', 'ResNet34_Weights', 'ResNet50_Weights', 'Wide_ResNet101_2_Weights', 'Wide_ResNet50_2_Weights', 'resnet', 'resnet101', 'resnet152', 'resnet18', 'resnet34', 'resnet50', 'wide_resnet101_2', 'wide_resnet50_2']


In [14]:
# 이미지 벡터화(resnet18를 이용한 임베딩)
model = resnet18(weights='ResNet18_Weights.DEFAULT')    # 이미 학습이 끝난 모델
model.eval()    # 모델을 평가 모드로 전환
model = torch.nn.Sequential(*(list(model.children()))[:-1])    # 맨 마지막 레이어는 뺀다 (512,) FC층은 제거(분류모델이 아니라 임베딩만 원함)
print(model)

# FC를 제외한 나머지 레이어들을 순차적으로 묶은 새로운 모델로 재구성
# image를 불러 Resnet18을 이용해 512 차원 벡터로 반환하는 함수
def image_to_vectorFunc(img_path):
  image = Image.open(img_path).convert('RGB')
  transform = transforms.Compose([
      transforms.Resize((224, 224)),    # Resnet18은 입력 이미지 크기 (3, 224, 224)를 원함
      transforms.ToTensor(),    # PIL.image를 torch.tensor로 변환
  ])
  tensor = transform(image).unsqueeze(0)    # 모델에 넣기 위해 batch 차원 추가. 앞에 +1 (unsqueeze는 차원 추가)

  with torch.no_grad():    # 역전파 계산을 끄고, 추론만 하겠다는 의미 (메모리 절약, 속도 증가)
    vec = model(tensor).squeeze().numpy()    # (512, ) 모든 크기의 1인 차원 제거
    # vec의 결과는 (1, 512, 1, 1) -> (512, )
  print(f"{img_path} -> 벡터(앞 10개) : {vec[:10]}")
  return vec.tolist()

# 이미지 경로

filenames = ['apple.jpg','banana.jpg','peach.jpeg']
image_files = [os.path.join('plc', name) for name in filenames]
print(image_files)
ids = [f"img{i}" for i in range(len(image_files))]
print(f"dis : {ids}")

# 벡터에 저장
client = PersistentClient("./image_chroma")
collection = client.get_or_create_collection("images")
for img_id, img_path in zip(ids, image_files):
  if not os.path.exists(img_path):
    print(f"파일 없음 : {img_path}")
    continue
  vec = image_to_vectorFunc(img_path)
  collection.add(
      embeddings=[vec],
      documents=[img_path],
      ids=[img_id],
      metadatas=[{"filename": img_path}]
  )

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [20]:
# 저장된 벡터 일부 출력
record = collection.get(ids=['img0'], include=['embeddings','documents','metadatas'])
print('id=img0')
print('documents : ', record['documents'])
print('metadatas : ', record['metadatas'])
print('embeddings : ', record['embeddings'][0][:10])

# 검색 이미지 설명
query_image_path = 'plc/apple.jpg'

if not os.path.exists(query_image_path):
  print(f"파일 없음 : {query_image_path}")
else:
  query_vec = image_to_vectorFunc(query_image_path)
  results = collection.query(
      query_embeddings=[query_vec],
      n_results=3,
      include=["documents", "metadatas", "distances"])

  print(f"검색 이미지 : {query_image_path}")
  print("유사 이미지")
  for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    print(f" - 파일명 : {meta['filename']} (유사도 거리:{dist:.4f})")

id=img0
documents :  ['plc/apple.jpg']
metadatas :  [{'filename': 'plc/apple.jpg'}]
embeddings :  [0.47947806 0.72125846 0.29288873 0.4093892  1.15829182 0.84491879
 0.22878729 1.3747462  0.94831544 0.82568812]
plc/apple.jpg -> 벡터(앞 10개) : [0.47947806 0.72125846 0.29288873 0.4093892  1.1582918  0.8449188
 0.22878729 1.3747462  0.94831544 0.8256881 ]
검색 이미지 : plc/apple.jpg
유사이미지
 - 파일명 : plc/apple.jpg (유사도 거리:0.0000)
 - 파일명 : plc/peach.jpeg (유사도 거리:185.0990)
 - 파일명 : plc/banana.jpg (유사도 거리:529.7032)
